# Phase 3 ROI Financial Engine

This notebook is the clean personal-project walkthrough for the Phase 3 decision-support model. It starts from the processed e-commerce tables, rebuilds the model artifacts, and reviews the investment scenarios used to prioritize markdown, checkout/OOS improvement, retention, and operational guardrails. DC rebalance is kept as a supporting diagnostic because its current net value is small.

## 1. Setup

The model code lives in `src/phase3_step1_models.py`. Running it will create both reusable model artifacts under `models/` and analysis-ready CSV outputs under `data/processed/`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

DATA_ROOT = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_ROOT / "processed"
MODEL_DIR = PROJECT_ROOT / "models"

PROJECT_ROOT

## 2. Rebuild The Phase 3 Model

The model is intentionally reproducible: baseline metrics are calculated from the existing processed tables, while scenario assumptions are exposed as configurable inputs instead of being hard-coded in the notebook.

In [ ]:
import argparse

from src.phase3_step1_models import run_phase3_step1

args = argparse.Namespace(
    project_root=PROJECT_ROOT,
    dataset_root=DATA_ROOT,
    investment_budget=500_000,
    holding_cost_rate=0.22,
    forward_holding_days=180,
    reverse_logistics_cost=20.0,
    min_transfer_cost=2.0,
    max_transfer_cost=8.0,
    max_rebalance_categories=12,
    models_dir=MODEL_DIR,
    force_refresh=False,
)

outputs = run_phase3_step1(args)
outputs

## 3. Baseline Metrics

These metrics anchor the scenario model. The important baseline problem is not direct COGS: gross margin is stable, while leakage, frozen inventory, returns, and low repeat behavior create operational drag.

In [ ]:
baseline = pd.read_json(MODEL_DIR / "baseline_metrics.json", typ="series")

baseline.loc[
    [
        "recognized_revenue",
        "recognized_gross_profit",
        "gross_margin_pct",
        "status_leakage_value",
        "frozen_inventory_value",
        "sell_through_pct",
        "current_return_rate",
        "valid_purchase_rate_pct",
        "repeat_rate_pct_of_valid",
    ]
]

## 4. Scenario ROI

The scenario layer compares pessimistic, base, and optimistic assumptions for a 500K investment envelope. The output is designed to be a benchmark for a later Excel or BI financial engine.

In [ ]:
scenario = pd.read_csv(PROCESSED_DIR / "phase3_scenario_output.csv")

scenario[
    [
        "scenario",
        "markdown_discount_pct",
        "total_incremental_gp",
        "investment_budget",
        "net_benefit_after_investment",
        "roi_pct",
        "payback_months",
    ]
]

## 5. Recommended Markdown Policy

Markdown recommendations are filtered by guardrails: the model avoids A-class items and keeps only rows with positive expected gross profit after markdown.

In [ ]:
markdown = pd.read_csv(PROCESSED_DIR / "phase3_markdown_recommendations.csv")

markdown.sort_values("net_incremental_value", ascending=False).head(10)[
    [
        "category",
        "department",
        "distribution_center_id",
        "abc_class",
        "recommended_discount_pct",
        "aged_180_units",
        "gross_profit_after_markdown",
        "holding_cost_saved",
        "net_incremental_value",
    ]
]

## 6. DC Rebalance Diagnostic

The current implementation uses a deterministic greedy min-cost transport fallback. It is useful as a diagnostic for stock-demand imbalance, but its current net value is too small to present as a primary ROI lever. Treat it as appendix/supporting evidence unless a later LP/MIP optimizer proves material value.

In [ ]:
dc_plan = pd.read_csv(PROCESSED_DIR / "phase3_dc_rebalance_plan.csv")

dc_plan.sort_values("net_value", ascending=False).head(10)[
    [
        "category",
        "from_dc",
        "to_dc",
        "transfer_units",
        "estimated_transfer_cost",
        "expected_protected_value",
        "net_value",
        "value_cost_ratio",
    ]
]

## 7. Model Quality Checks

This section keeps the caveats visible: price elasticity is a proxy because explicit discount history is missing, OOS uplift is scenario-based, DC rebalance is a low-materiality heuristic, and retention is a supporting module rather than a standalone churn model.

In [ ]:
quality = pd.read_csv(PROCESSED_DIR / "phase3_model_quality_scorecard.csv")
evaluation = pd.read_csv(PROCESSED_DIR / "phase3_model_evaluation.csv")

display(quality)
display(evaluation[["check_name", "status", "detail"]])

## 8. Portfolio Takeaway

The base case is strong enough to justify Phase 3 as a decision-support layer: markdown and operational fixes can generate material incremental gross profit, but pessimistic ROI remains thin. The portfolio story should lead with inventory markdown guardrails and checkout/OOS recovery, keep DC rebalance secondary, and use phase gates, kill-switches, and forecast monitoring before scaling the full budget.